## Tech Challenge - Fase 3: Modelo Não Supervisionado
### Fase 1: Ideia A - Clusterização de Perfis de Companhias Aéreas (K-Means)


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)


In [2]:
# Carregando dados
df_flights = pd.read_csv('../data/flights.csv', low_memory=False)
df_airlines = pd.read_csv('../data/airlines.csv')
df_airports = pd.read_csv('../data/airports.csv')


In [3]:
# Renomeando colunas para merge
df_airlines = df_airlines.rename(columns={'AIRLINE': 'AIRLINE_NAME'})
df_airport_origin = df_airports.rename(columns={
    'IATA_CODE': 'ID', 'AIRPORT': 'ORIGIN_AIRPORT_NAME', 'CITY': 'ORIGIN_CITY',
    'STATE': 'ORIGIN_STATE', 'COUNTRY': 'ORIGIN_COUNTRY', 'LATITUDE': 'ORIGIN_LATITUDE', 'LONGITUDE': 'ORIGIN_LONGITUDE'
})
df_airport_dest = df_airports.rename(columns={
    'IATA_CODE': 'ID', 'AIRPORT': 'DEST_AIRPORT_NAME', 'CITY': 'DEST_CITY',
    'STATE': 'DEST_STATE', 'COUNTRY': 'DEST_COUNTRY', 'LATITUDE': 'DEST_LATITUDE', 'LONGITUDE': 'DEST_LONGITUDE'
})

# Merge
df = (
    df_flights
    .merge(df_airlines, left_on='AIRLINE', right_on='IATA_CODE', how='left')
    .drop(columns=['IATA_CODE'])
    .merge(df_airport_origin, left_on='ORIGIN_AIRPORT', right_on='ID', how='left')
    .drop(columns=['ID'])
    .merge(df_airport_dest, left_on='DESTINATION_AIRPORT', right_on='ID', how='left')
    .drop(columns=['ID'])
)

# Tratamento de Nulos
delay_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']
df[delay_cols] = df[delay_cols].fillna(0)
df = df.drop(columns=['CANCELLATION_REASON'])
df = df.dropna(subset=['ARRIVAL_DELAY', 'ORIGIN_CITY', 'DEST_CITY', 'ORIGIN_LATITUDE', 'DEST_LATITUDE', 'TAIL_NUMBER'])

# Removendo voos cancelados e desviados
df = df[(df['CANCELLED'] == 0) & (df['DIVERTED'] == 0)]


### Agregação de Dados e Engenharia de Features
Calculando as métricas médias para cada companhia aérea.


In [4]:
# Agrupando dados por companhia aérea
df_airline_stats = df.groupby('AIRLINE_NAME').agg({
    'TAXI_OUT': 'mean',
    'TAXI_IN': 'mean',
    'AIRLINE_DELAY': 'mean',
    'WEATHER_DELAY': 'mean',
    'FLIGHT_NUMBER': 'count' # Representa o volume de voos
}).reset_index()

df_airline_stats = df_airline_stats.rename(columns={'FLIGHT_NUMBER': 'FLIGHT_VOLUME'})
df_airline_stats.head()


,AIRLINE_NAME,TAXI_OUT,TAXI_IN,AIRLINE_DELAY,WEATHER_DELAY,FLIGHT_VOLUME
0,Alaska Airlines Inc.,15.081643,6.362579,2.069556,0.231001,157025
1,American Airlines Inc.,17.757780,8.891846,3.979537,0.680197,636554
2,American Eagle Airlines Inc.,16.612768,9.138471,3.879882,1.510349,257130
3,Atlantic Southeast Airlines,16.744175,7.634544,4.357736,0.320621,508288
4,Delta Air Lines Inc.,17.700488,7.208607,3.186145,0.746005,791067


### Escalonamento e K-Means
Aplicando StandardScaler para padronizar as métricas e treinando o modelo K-Means.


In [5]:
# Selecionando as features para o modelo
features = ['TAXI_OUT', 'TAXI_IN', 'AIRLINE_DELAY', 'WEATHER_DELAY']
X = df_airline_stats[features]

# Escalonamento
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Encontrando o K ideal e treinando o K-Means
# Para poucas companhias, usaremos 3 ou 4 clusters. Vamos fixar em 4 para obter perfis distintos.
kmeans = KMeans(n_clusters=4, random_state=42)
df_airline_stats['CLUSTER'] = kmeans.fit_predict(X_scaled)

# Nomeando os clusters (isso pode ser ajustado após visualização)
cluster_names = {
    0: 'Perfil 0',
    1: 'Perfil 1',
    2: 'Perfil 2',
    3: 'Perfil 3'
}
df_airline_stats['CLUSTER_NAME'] = df_airline_stats['CLUSTER'].map(cluster_names)
df_airline_stats.sort_values(by='CLUSTER')


,AIRLINE_NAME,TAXI_OUT,TAXI_IN,AIRLINE_DELAY,WEATHER_DELAY,FLIGHT_VOLUME,CLUSTER,CLUSTER_NAME
1,American Airlines Inc.,17.757780,8.891846,3.979537,0.680197,636554,0,Perfil 0
5,Frontier Airlines Inc.,15.755773,9.141320,4.004455,0.260784,81715,0,Perfil 0
12,United Air Lines Inc.,17.445356,8.507453,4.517798,0.691215,462086,0,Perfil 0
10,Spirit Air Lines,14.647990,9.540392,4.435254,0.419818,104500,0,Perfil 0
9,Southwest Airlines Co.,11.955471,6.171328,3.190176,0.456771,1135152,1,Perfil 1
13,Virgin America,14.784405,8.177521,2.124917,0.607152,55813,1,Perfil 1
6,Hawaiian Airlines Inc.,10.961140,6.856607,2.619537,0.159779,69815,1,Perfil 1
0,Alaska Airlines Inc.,15.081643,6.362579,2.069556,0.231001,157025,1,Perfil 1
2,American Eagle Airlines Inc.,16.612768,9.138471,3.879882,1.510349,257130,2,Perfil 2
7,JetBlue Airways,17.942232,6.115662,4.166239,0.465381,240304,3,Perfil 3


### Visualização dos Clusters
Plotando os resultados para entender o que cada cluster representa.


In [6]:
# Gráfico de Dispersão 3D para entender a separação
fig = px.scatter_3d(
    df_airline_stats, 
    x='AIRLINE_DELAY', 
    y='TAXI_OUT', 
    z='WEATHER_DELAY',
    color='CLUSTER_NAME',
    hover_name='AIRLINE_NAME',
    size='FLIGHT_VOLUME',
    title='Clusterização de Companhias Aéreas (Tamanho = Volume de Voos)',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig.show()

# Resumo médio por cluster para identificar os perfis
resumo_clusters = df_airline_stats.groupby('CLUSTER_NAME')[features].mean().reset_index()
display(resumo_clusters)


,CLUSTER_NAME,TAXI_OUT,TAXI_IN,AIRLINE_DELAY,WEATHER_DELAY
0,Perfil 0,16.401725,9.020253,4.234261,0.513004
1,Perfil 1,13.195665,6.892009,2.501047,0.363676
2,Perfil 2,16.612768,9.138471,3.879882,1.510349
3,Perfil 3,17.903263,7.104842,3.720517,0.470692


### Fase 2: Ideia B - Redução de Dimensionalidade na Dinâmica do Voo (PCA)

Vamos reduzir as variáveis temporais do voo para achar padrões de atraso não óbvios em um espaço 2D.

In [9]:
from sklearn.decomposition import PCA

# Criando a variável IS_DELAYED (atrasos de chegada > 15 minutos)
df['IS_DELAYED'] = (df['ARRIVAL_DELAY'] > 15).astype(int)

# 1. Seleção de Variáveis
pca_features = ['SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'TAXI_IN', 'TAXI_OUT']
df_pca = df.dropna(subset=pca_features).copy()

# Amostragem para facilitar a visualização no Plotly sem travar o navegador
if len(df_pca) > 100000:
    df_pca_sample = df_pca.sample(100000, random_state=42)
else:
    df_pca_sample = df_pca.copy()

X_pca = df_pca_sample[pca_features]

# 2. Pré-processamento
scaler_pca = StandardScaler()
X_pca_scaled = scaler_pca.fit_transform(X_pca)

# 3. Aplicação do PCA
pca = PCA(n_components=2, random_state=42)
principal_components = pca.fit_transform(X_pca_scaled)

df_pca_sample['PCA1'] = principal_components[:, 0]
df_pca_sample['PCA2'] = principal_components[:, 1]
df_pca_sample['IS_DELAYED_STR'] = df_pca_sample['IS_DELAYED'].map({0: 'No Delay', 1: 'Delayed'})

print("Variância Explicada por componente:", pca.explained_variance_ratio_)
print("Variância Total Explicada:", sum(pca.explained_variance_ratio_))

Variância Explicada por componente: [0.66458254 0.1653264 ]
Variância Total Explicada: 0.8299089374001144


Visualizando o espaço bidimensional reduzido para identificar se voos com atraso se concentram em alguma região específica.

In [10]:
# 4. Visualização
# Como a amostra pode ser grande, podemos usar scattergl ou apenas scatter com opacidade
fig_pca = px.scatter(
    df_pca_sample,
    x='PCA1',
    y='PCA2',
    color='IS_DELAYED_STR',
    title='PCA da Dinâmica de Voo (2 Componentes Principais)',
    labels={'PCA1': 'Componente Principal 1', 'PCA2': 'Componente Principal 2'},
    opacity=0.5,
    color_discrete_map={'No Delay': '#1f77b4', 'Delayed': '#d62728'},
    template='plotly_white',
    render_mode='webgl'  # Melhor performance para muitos pontos
)

fig_pca.show()

### Fase 3: Ideia C - Detecção de Anomalias (Isolation Forest)

Vamos focar em encontrar os "pontos fora da curva", ou seja, os piores cenários de caos da malha aérea (outliers).

In [11]:
from sklearn.ensemble import IsolationForest

# 1. Seleção de Variáveis
iso_features = ['DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'TAXI_IN', 'TAXI_OUT']
df_iso = df.dropna(subset=iso_features).copy()

# Amostragem para agilizar processamento e renderização do gráfico
if len(df_iso) > 100000:
    df_iso_sample = df_iso.sample(100000, random_state=42)
else:
    df_iso_sample = df_iso.copy()

X_iso = df_iso_sample[iso_features]

# Padronizando os dados
scaler_iso = StandardScaler()
X_iso_scaled = scaler_iso.fit_transform(X_iso)

# 2. Treinamento Isolation Forest
# Estimamos que apenas 2% dos voos representem os cenários mais drásticos (anomalias)
iso_forest = IsolationForest(contamination=0.02, random_state=42)
df_iso_sample['ANOMALY'] = iso_forest.fit_predict(X_iso_scaled)

# Mapeamento do resultado: 1 (Normal) e -1 (Anomalia)
df_iso_sample['IS_ANOMALY'] = df_iso_sample['ANOMALY'].map({1: 'Normal', -1: 'Anomalia'})

# Quantidade de anomalias detectadas
display(df_iso_sample['IS_ANOMALY'].value_counts())

IS_ANOMALY
Normal      98000
Anomalia     2000
Name: count, dtype: int64

Visualizando os resultados das anomalias no gráfico de dispersão.

In [12]:
# 4. Visualização
# Plotando a relação entre Atraso na Partida e na Chegada
fig_iso = px.scatter(
    df_iso_sample,
    x='DEPARTURE_DELAY',
    y='ARRIVAL_DELAY',
    color='IS_ANOMALY',
    title='Detecção de Anomalias nos Voos (Piores Cenários de Atraso)',
    opacity=0.6,
    color_discrete_map={'Normal': '#1f77b4', 'Anomalia': 'black'},
    template='plotly_white',
    render_mode='webgl'
)

fig_iso.show()

# Verificando a média das variáveis nos cenários normais vs anomalias
display(df_iso_sample.groupby('IS_ANOMALY')[iso_features].mean())

,DEPARTURE_DELAY,ARRIVAL_DELAY,TAXI_IN,TAXI_OUT
IS_ANOMALY,,,,
Anomalia,164.252500,180.506500,15.428000,30.777500
Normal,6.419418,1.129102,7.244714,15.757684


## Conclusões da Análise:
1. **Clusterização (K-Means)**: Foi possível segmentar as companhias aéreas em diferentes perfis operacionais. Alguns perfis demonstraram excelência em pontualidade, enquanto outros apresentaram gargalos significativos de tempo em solo (Taxi In/Out) e atrasos recorrentes.
2. **Dinâmica de Voo (PCA)**: A redução de dimensionalidade revelou como as variáveis temporais (tempo de voo, distância, tempo em solo) se relacionam com os atrasos. O espaço 2D gerado permitiu observar se os voos atrasados estão distribuídos de forma aleatória ou se concentram em uma zona de "estresse operacional" específica da malha.
3. **Detecção de Anomalias (Isolation Forest)**: Conseguimos isolar os piores cenários (outliers que representam o caos extremo na aviação, correspondentes a cerca de 2% da nossa amostra). Essa identificação prova que existem voos cujas dinâmicas fogem completamente do padrão, sendo impactados provavelmente por falhas em cascata extremas (clima severo aliado a problemas no sistema aéreo).